# Access Modifiers, Encapsulation, and Name Mangling in Python

**Session:** 2026-11-01 · OOP with Python  
**Module:** 01 · Python Fundamentals & Algorithmic Thinking

Encapsulation is the object-oriented principle of bundling state data and manipulating methods within a single modular unit, while restricting direct external access to internal implementation details. 

Unlike languages such as C++ or Java that enforce access boundaries via strict compiler keywords (`public`, `protected`, `private`), Python operates under the pragmatic design philosophy that **"we are all consenting adults here"**. Instead of rigid compiler restrictions, Python employs syntactic conventions (`_`) and compiler-level name transformations known as **name mangling** (`__`).

## Learning Outcomes

- Understand the distinctions between Public, Protected, and Private access tiers in Python.
- Analyze the internal mechanics of compiler name mangling (`_ClassName__attribute`) using `__dict__`.
- Trace attribute access and visibility across multi-level inheritance hierarchies (`Person` -> `Student` -> `GraduateStudent`).
- Implement robust, validated domain state using Pythonic `@property` getter and setter descriptors.
- Ingest and encapsulate structured organization records from `data/staff_hierarchy.json`.

## 1. Public Access Modifiers

In Python, all class members (attributes and methods) are **public by default**. Any code outside the class can read and mutate public attributes directly without restriction.

In [ ]:
class PublicStudent:
    """Demonstrating standard public attributes."""

    def __init__(self, student_id: str, name: str) -> None:
        self.student_id = student_id
        self.name = name

    def get_name(self) -> str:
        return self.name


student = PublicStudent(student_id="STU-001", name="Alex Morgan")

# Unrestricted public access
print("Direct public read:", student.name)
print("Public method call:", student.get_name())

# Direct public mutation
student.name = "Alexandria Morgan"
print("Mutated public name:", student.name)

## 2. Protected Access Modifiers (`_single_underscore`)

In Python, an attribute or method prefixed with a single underscore `_variable` is designated as **protected by convention**.

- **Syntactic meaning:** It signals to fellow engineers and static analysis tools that the member is part of the class's internal implementation and should not be accessed directly from external client code.
- **Inheritance scope:** Protected members are explicitly intended to be accessed and overridden by subclasses.
- **Runtime enforcement:** Python's runtime does *not* raise an error if external code accesses a single-underscore attribute. The protection is an advisory contract.

In [ ]:
class Person:
    """Base person with protected identity attributes."""

    def __init__(self, person_id: str, name: str) -> None:
        # Protected attributes (advisory contract)
        self._person_id = person_id
        self._name = name

    def _display_identity(self) -> str:
        """Protected helper method intended for internal and subclass usage."""
        return f"ID: {self._person_id} | Name: {self._name}"


class Student(Person):
    """Derived student class accessing protected superclass members."""

    def __init__(self, person_id: str, name: str, major: str) -> None:
        super().__init__(person_id, name)
        self._major = major

    def get_student_summary(self) -> str:
        # Subclass legitimately accesses protected member of parent class
        identity = self._display_identity()
        return f"{identity} | Major: {self._major}"


student = Student("STU-101", "Julian Rivera", "Computer Systems")
print(student.get_student_summary())

## 3. Private Access Modifiers and Name Mangling (`__double_underscore`)

When an attribute or method is prefixed with a double leading underscore (without trailing double underscores), Python activates **name mangling**:

$$\text{\texttt{\_\_attribute}} \longrightarrow \text{\texttt{\_ClassName\_\_attribute}}$$

### Why Name Mangling Exists
Name mangling was **not** created to simulate strict privacy or security; it was designed to prevent **accidental namespace collisions in inheritance hierarchies**. If a parent class uses `__internal_state` and a derived subclass inadvertently defines `__internal_state`, each class binds to a different mangled identifier (`_Parent__internal_state` vs `_Child__internal_state`), preventing silent state corruption.

In [ ]:
class PrivateAccount:
    """Demonstrating private members and runtime name mangling."""

    def __init__(self, account_holder: str, initial_balance: float) -> None:
        self.account_holder = account_holder
        # Private attribute: triggers name mangling
        self.__balance = float(initial_balance)

    def get_balance(self) -> float:
        """Controlled public accessor."""
        return self.__balance

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Deposit must be positive.")
        self.__balance += amount


account = PrivateAccount("Dr. Eleanor Vance", 15000.0)

# 1. Direct access via original identifier raises AttributeError:
try:
    print(account.__balance)
except AttributeError as e:
    print("Direct private access rejected as expected:", e)

# 2. Inspecting the object's __dict__ reveals the mangled identifier:
print("\nObject attribute namespace (__dict__):")
for k, v in account.__dict__.items():
    print(f"  {k} -> {v}")

# 3. Access is still technically possible via the mangled name (proving no compiler sandbox):
mangled_key = "_PrivateAccount__balance"
print(f"\nMangled attribute access ({mangled_key}): ${getattr(account, mangled_key):,.2f}")

## 4. Multi-Level Inheritance with Access Modifiers

Let's examine how protected and private members behave across a three-tier hierarchy:
$$\text{Person} \longrightarrow \text{Student} \longrightarrow \text{GraduateStudent}$$

We de-brand and formalize the lecture slide example:
- `Student` defines protected `_student_id`, `_name`, and protected `_display_information()`.
- `GraduateStudent` inherits from `Student`, introduces specialized `_research_field`, and invokes the protected parent method.

In [ ]:
class AcademicStudent:
    """Superclass defining protected data members and methods."""

    def __init__(self, student_id: int, name: str) -> None:
        # Protected members
        self._id = student_id
        self._name = name

    def _display_information(self) -> None:
        """Protected member function."""
        print(f"  [Student] ID: {self._id} | Name: {self._name}")


class GraduateStudent(AcademicStudent):
    """Derived class extending Student with graduate research metadata."""

    def __init__(self, student_id: int, name: str, research_field: str) -> None:
        super().__init__(student_id, name)
        # Private member specific to GraduateStudent
        self.__research_field = research_field

    def display_details(self) -> None:
        """Public method calling protected superclass method."""
        print("Graduate Scholar Academic Dossier:")
        # Accessing protected superclass method
        self._display_information()
        # Accessing own private member
        print(f"  [Research] Specialized Field: {self.__research_field}")


grad = GraduateStudent(
    student_id=23032023, name="Sophia Zhang", research_field="Self-Supervised Foundation Models"
)
grad.display_details()

## 5. Pythonic Encapsulation with `@property`

The idiomatic Python alternative to Java-style getter and setter methods (`get_salary()`, `set_salary()`) is the `@property` decorator.

Properties allow methods to be accessed with attribute-style syntax (`obj.salary`) while retaining the ability to execute validation logic, type checks, and invariant guards on assignment.

In [ ]:
class AcademicStaff:
    """Encapsulated staff record with validated properties."""

    def __init__(
        self,
        staff_id: str,
        name: str,
        role: str,
        department: str,
        base_salary: float,
        bonus_rate: float,
    ) -> None:
        self.staff_id = staff_id
        self.name = name
        self.role = role
        self.department = department
        # Use property setters during initialization to enforce validation
        self.base_salary = base_salary
        self.bonus_rate = bonus_rate

    @property
    def base_salary(self) -> float:
        """Getter for monthly base salary."""
        return self._base_salary

    @base_salary.setter
    def base_salary(self, value: float) -> None:
        """Setter validating that base salary is strictly positive."""
        if not isinstance(value, (int, float)):
            raise TypeError("Salary must be numeric.")
        if value <= 0:
            raise ValueError(f"Base salary must be positive, got {value}.")
        self._base_salary = float(value)

    @property
    def bonus_rate(self) -> float:
        """Getter for annual bonus multiplier."""
        return self._bonus_rate

    @bonus_rate.setter
    def bonus_rate(self, value: float) -> None:
        """Setter validating that bonus rate is between 0.0 and 1.0."""
        if not (0.0 <= value <= 1.0):
            raise ValueError(f"Bonus rate must be within [0.0, 1.0], got {value}.")
        self._bonus_rate = float(value)

    @property
    def total_annual_compensation(self) -> float:
        """Computed property combining base salary and annual performance bonus."""
        annual_base = self._base_salary * 12.0
        return annual_base * (1.0 + self._bonus_rate)


staff_member = AcademicStaff(
    staff_id="FAC-101",
    name="Dr. Eleanor Vance",
    role="Faculty",
    department="Machine Learning",
    base_salary=9500.0,
    bonus_rate=0.25,
)

print(f"Staff: {staff_member.name}")
print(f"Monthly Base: ${staff_member.base_salary:,.2f}")
print(f"Annual Total Compensation: ${staff_member.total_annual_compensation:,.2f}")

# Verification of invariant enforcement:
try:
    staff_member.base_salary = -500.0
except ValueError as e:
    print("\nValidation guard caught negative salary:", e)

try:
    staff_member.bonus_rate = 1.75
except ValueError as e:
    print("Validation guard caught invalid bonus rate:", e)

## 6. Parsing and Encapsulating Institutional Datasets

We now load `data/staff_hierarchy.json`, transform each raw record into an encapsulated `AcademicStaff` instance, and compute departmental compensation analytics.

In [ ]:
import json
from pathlib import Path

staff_file = Path("../data/staff_hierarchy.json")
if not staff_file.exists():
    staff_file = Path("data/staff_hierarchy.json")

with staff_file.open("r", encoding="utf-8") as f:
    roster_data = json.load(f)

institution = roster_data["institution"]
records = [
    AcademicStaff(
        staff_id=r["id"],
        name=r["name"],
        role=r["role"],
        department=r["department"],
        base_salary=r["base_salary"],
        bonus_rate=r["bonus_rate"],
    )
    for r in roster_data["staff_records"]
]

print(f"Institution: {institution}")
print(f"Total Staff Processed: {len(records)}\n")
print(f"{'ID':8s} | {'Name':20s} | {'Department':22s} | {'Annual Compensation':20s}")
print("-" * 78)
for member in records:
    print(
        f"{member.staff_id:8s} | {member.name:20s} | {member.department:22s} | ${member.total_annual_compensation:18,.2f}"
    )